# SLIIT IT3051 - Data Mining & Predictive Analytics
## EV Battery Prognostics & Health Management (PHM) Dual-Task System
### Group: Necrons | Phase 1: EDA, Data Cleaning & Preprocessing (Week 1 / Viva 1)

---

### Team Roles & Modular Work Breakdown:
| Person | Topic Owned | Key Deliverables |
| :--- | :--- | :--- |
| **Person A** | **Data Structure, Schema & Missingness Audit** | Schema inspection, variable types, duplicate checks, 67-column missingness breakdown, sensor sanity screening |
| **Person B** | **Distributions, Outliers & Treatment Decisions** | Univariate distributions, RUL bell-curve analysis, IQR/Z-score outlier detection, physical outlier justification |
| **Person C** | **Imbalance, Multicollinearity & Feature Engineering** | Class imbalance (18,616 vs 1,384), correlation heatmaps, bivariate degradation analysis, 4 domain features |
| **Person D** | **Data Leakage Guard, Splits & ColumnTransformer** | Target Isolation, ID exclusion, stratified train/test split, production ColumnTransformer pipeline |


### 0. Environment Setup & Global Imports


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'

print('Libraries loaded successfully!')


---
# Section B: Distributions, Skewness, Outliers & Treatment Decisions
**Owner: Person B**
**Focus:** Continuous feature distributions, Task 1 target distribution, outlier detection via IQR & Z-score, and domain justification for outlier treatment.


#### B.1 Task 1 Target: Remaining Life Cycles Distribution


In [ ]:
# Task 1: RUL Distribution Analysis
rul_data = df_raw['predicted_remaining_life_cycles'].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(rul_data, kde=True, color='#2b5c8f', bins=40, ax=ax)
ax.axvline(rul_data.mean(), color='#e63946', linestyle='--', linewidth=2, label=f'Mean: {rul_data.mean():.1f}')
ax.axvline(rul_data.median(), color='#2a9d8f', linestyle=':', linewidth=2, label=f'Median: {rul_data.median():.1f}')
ax.set_title('Task 1 Target Distribution: Predicted Remaining Life Cycles', fontsize=12, fontweight='bold')
ax.set_xlabel('Remaining Life Cycles')
ax.set_ylabel('Record Frequency')
ax.legend()
plt.show()

print(f'RUL Mean:   {rul_data.mean():.2f}')
print(f'RUL Median: {rul_data.median():.2f}')
print(f'RUL Std:    {rul_data.std():.2f}')
print(f'RUL Skew:   {rul_data.skew():.3f} (Symmetric Bell Curve)')


#### B.2 Key Physical Degradation Distributions


In [ ]:
# Physical sensor distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df_raw['cell_voltage_avg'], kde=True, ax=axes[0, 0], color='#264653')
axes[0, 0].set_title('Average Cell Voltage (V)', fontweight='bold')

sns.histplot(df_raw['pack_voltage'], kde=True, ax=axes[0, 1], color='#2a9d8f')
axes[0, 1].set_title('Pack Voltage (V)', fontweight='bold')

sns.histplot(df_raw['cell_temperature_max'], kde=True, ax=axes[1, 0], color='#e76f51')
axes[1, 0].set_title('Max Cell Temperature (C)', fontweight='bold')

sns.histplot(df_raw['internal_resistance'], kde=True, ax=axes[1, 1], color='#e63946')
axes[1, 1].set_title('Internal Resistance (Ohm)', fontweight='bold')

plt.tight_layout()
plt.show()


#### B.3 Outlier Detection & Outlier Treatment Decision


In [ ]:
# IQR & Z-score Outlier Audit
def audit_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers_iqr = ((series < lower_bound) | (series > upper_bound)).sum()
    
    mean = series.mean()
    std = series.std()
    outliers_z = (np.abs((series - mean) / std) > 3).sum()
    
    return outliers_iqr, outliers_z, lower_bound, upper_bound

key_cols = ['internal_resistance', 'cell_temperature_max', 'voltage_imbalance', 'capacity_loss_percent']
print('=== OUTLIER DETECTION AUDIT (IQR & Z-SCORE) ===')
for col in key_cols:
    s = df_raw[col].dropna()
    o_iqr, o_z, lb, ub = audit_outliers(s)
    print(f'{col}: IQR Outliers={o_iqr:,} (bounds: [{lb:.2f}, {ub:.2f}]), Z-score (|z|>3)={o_z:,}')


**Person B Outlier Treatment Decision (Viva Justification):**
- **Decision:** Outliers are **retained** rather than naively truncated or deleted.
- **Physical Justification:** High temperature spikes (> 60 C), elevated internal resistance (> 0.8 Ohm), and high voltage imbalance are **true electro-thermal degradation precursors**. If we drop them, we remove the most critical failure signals from Task 2!
- We employ **StandardScaler** and tree-based ensembles (Random Forest / XGBoost), which are naturally robust to monotonic outliers.


---
# Section C: Class Imbalance, Multicollinearity & Feature Engineering
**Owner: Person C**
**Focus:** Task 2 binary class imbalance, Pearson correlation analysis, physical discrimination boxplots, and Stage 3 domain feature engineering.
